[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivastar/ml_for_astro_notebooks/blob/main/session2b_solutions.ipynb)

# Session 3b — Galaxies and their Environment in the Coma Supercluster

In the last notebook we clustered simulated data, where we knew the answer. Now we apply the same tools — Gaussian mixtures, BIC and silhouette, bootstrap stability, and DBSCAN — to real galaxies.

**The data.** About 3700 SDSS DR17 galaxies with spectroscopic redshifts in a thin slice, $0.018 < z < 0.028$, around the **Coma supercluster**. The slice contains two rich clusters, **Coma** (Abell 1656) and **Abell 1367**, the filament between them, smaller groups, and many isolated field galaxies.

**The idea.** Galaxies in dense structures tend to have the same colour: they have stopped forming stars and sit on the *red sequence*. So instead of clustering on positions, we describe every galaxy by **two properties**:

* its colour, $g - r$,
* its **local environment**: the surface density of neighbours, measured from the distance to its Nth nearest neighbour.

We then look for populations in this colour–density space, without telling the algorithm where Coma is. Afterwards we check whether the populations we find mean something **on the sky** — a property we did *not* cluster on.

Cells marked **✏️ Exercise** are for you to complete.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q astropy gdown

import warnings
from pathlib import Path

import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.cosmology import Planck18 as cosmo

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [ ]:
#@title Setup (run me) {display-mode: "form"}

plt.rcParams.update({
    "figure.figsize": (7, 5), "figure.dpi": 110, "font.size": 12, "axes.titlesize": 13,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "lines.linewidth": 2, "legend.frameon": False,
})

# The two rich clusters in the slice (positions for plotting and validation only)
CLUSTERS = {"Coma": (194.95, 27.98), "Abell 1367": (176.19, 19.70)}

def sky_axes(ax, title=""):
    ax.set(xlim=(200.5, 169.5), ylim=(14.5, 35.5), xlabel="RA [deg]", ylabel="Dec [deg]", title=title)
    ax.set_aspect(1/np.cos(np.deg2rad(25)))
    for name, (ra, dec) in CLUSTERS.items():
        ax.annotate(name, (ra, dec), xytext=(0, 14), textcoords="offset points", ha="center",
                    fontsize=10, color="black", fontweight="bold")

## 1. The data

The catalogue is downloaded once and cached in a local direcotry.

* `z BETWEEN 0.018 AND 0.028` — a thin redshift slice around Coma ($z = 0.023$)
* `petroMag_r - extinction_r < 17.77` — the SDSS main-sample magnitude limit, so the sample has a uniform depth
* `zWarning = 0` — only secure redshifts

In [ ]:
FILE_ID = "1EiuoaYAoMAx2ioldqcreWPCYMbrvrl0d"
DATA = Path("./") / "coma_slice_dr17.fits"

if not DATA.exists():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=FILE_ID, output=str(DATA), quiet=False)
else:
    print(f"Using cached catalogue: {DATA}  ({DATA.stat().st_size/1e6:.1f} MB)")

d = Table.read(DATA).to_pandas().drop_duplicates("objID").reset_index(drop=True)
print(f"{len(d)} galaxies")

# Galactic-extinction-corrected colour and absolute magnitude
# (at z ~ 0.02 the k-corrections are only a few hundredths of a magnitude, so we neglect them)
d["gr"] = (d.modelMag_g - d.extinction_g) - (d.modelMag_r - d.extinction_r)
d["Mr"] = d.petroMag_r - d.extinction_r - cosmo.distmod(d.z.to_numpy()).value
d[["ra", "dec", "z", "petroMag_r", "gr", "Mr"]].describe().T[["mean", "min", "max"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), gridspec_kw=dict(width_ratios=[1.35, 1, 1]))
axes[0].scatter(d.ra, d.dec, s=3, color="gray", alpha=0.6, linewidth=0)
sky_axes(axes[0], "The slice on the sky")

axes[1].hist(d.z, bins=40)
axes[1].set(xlabel="redshift", ylabel="galaxies", title="Redshift distribution")

axes[2].scatter(d.Mr, d.gr, s=3, color="gray", alpha=0.5, linewidth=0)
axes[2].set(xlim=(-16.5, -23.5), ylim=(-0.2, 1.4), xlabel=r"$M_r$", ylabel="g − r",
            title="Colour–magnitude diagram")
fig.tight_layout()

bad = ~d.gr.between(0, 1.0)
print(f"{bad.sum()} galaxies with g − r outside [0, 1]:")
print(d.loc[bad, ["ra", "dec", "modelMag_g", "modelMag_r", "gr"]].round(2).to_string())

**Look before you cluster.**

* The colour–magnitude diagram already shows two populations: a tight **red sequence** at $g - r \approx 0.75$ and a broad **blue cloud**.
* A handful of galaxies have unphysical colours ($g - r$ up to 7!). These are photometry failures — typically large, bright galaxies that SDSS "shredded" into pieces, or objects next to bright stars. A Gaussian mixture will happily spend a whole component on a few outliers like these, so we remove them. With real data, this step is never optional.

In [ ]:
d = d[d.gr.between(0, 1.0)].reset_index(drop=True)
print(f"{len(d)} galaxies after removing colour outliers")

## 2. A density feature from the N-th nearest neighbour

A standard measure of a galaxy's environment is the **projected surface density** of its neighbours,

$$\Sigma_N = \frac{N}{\pi\, r_N^2},$$

where $r_N$ is the projected distance to the Nth nearest neighbour **within the redshift slice**. We use $N = 5$.

This is exactly the calculation behind the **k-distance plot** in the last notebook — we are just using it as a feature instead of for choosing `eps`.

Two practical details:
* Distances on the sky should be computed on the sphere. `NearestNeighbors(metric="haversine")` does this: it takes (Dec, RA) in **radians** and returns angular distances in radians.
* An angle $\theta$ corresponds to a projected comoving distance $r = D_C\,\theta$, where $D_C$ is the comoving distance to the slice. That way $\Sigma_N$ is in physical units ($\mathrm{Mpc}^{-2}$).

### ✏️ Exercise 1 — Compute $\Sigma_5$

1. Build the array `coords` of shape $(N_\mathrm{gal}, 2)$ holding (Dec, RA) in radians.
2. Fit `NearestNeighbors(n_neighbors=N_NEIGH + 1, metric="haversine")` and get the distances with `.kneighbors(coords)`. Why `N_NEIGH + 1`?
3. Convert the distance to the $N$th neighbour into Mpc (`r_N`), and store $\log_{10}\Sigma_N$ in `d["logSigma"]`.

In [ ]:
N_NEIGH = 5
D_C = cosmo.comoving_distance(0.023).value          # Mpc, comoving distance to the slice

### BEGIN SOLUTION
coords = np.deg2rad(d[["dec", "ra"]].to_numpy())
dist, _ = NearestNeighbors(n_neighbors=N_NEIGH + 1, metric="haversine").fit(coords).kneighbors(coords)
r_N = dist[:, N_NEIGH] * D_C                        # column 0 is the galaxy itself
d["logSigma"] = np.log10(N_NEIGH / (np.pi * r_N**2))
### END SOLUTION

print(f"D_C = {D_C:.1f} Mpc;  1 deg = {D_C*np.pi/180:.2f} Mpc")
print(d.logSigma.describe().round(2))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw=dict(width_ratios=[1.35, 1]))
order = np.argsort(d.logSigma.to_numpy())                     # draw the densest on top
sc = ax1.scatter(d.ra.to_numpy()[order], d.dec.to_numpy()[order], c=d.logSigma.to_numpy()[order],
                 s=5, cmap="Blues", vmin=-0.5, vmax=2.5, linewidth=0)
sky_axes(ax1, r"Galaxies coloured by $\log\,\Sigma_5$")
fig.colorbar(sc, ax=ax1, label=r"$\log\,\Sigma_5$ [Mpc$^{-2}$]", shrink=0.85)

ax2.hexbin(d.logSigma, d.gr, gridsize=40, bins="log", mincnt=1, cmap="Blues",
           extent=(-1, 3, 0, 1))
ax2.set(xlabel=r"$\log\,\Sigma_5$ [Mpc$^{-2}$]", ylabel="g − r", title="Colour vs. environment")
fig.tight_layout();

## 3. Populations in colour–density space

Our feature vector is $(g - r,\ \log\Sigma_5)$. The two features have **different units**, so we standardise them — exactly as in the simulated-data notebook. (Note that we take the logarithm of the density first: $\Sigma_5$ spans three orders of magnitude.)

In [ ]:
FEATURES = ["gr", "logSigma"]
scaler = StandardScaler().fit(d[FEATURES])
X = scaler.transform(d[FEATURES])
print(X.shape, "  means:", X.mean(axis=0).round(3), "  stds:", X.std(axis=0).round(3))

### ✏️ Exercise 2 — How many populations?

For $k = 1, \dots, 7$, fit `GaussianMixture(n_components=k, n_init=5, random_state=0)` to `X`. Store the BIC in `bics` and, for $k \ge 2$, the silhouette score of the predicted labels in `sils` (use `np.nan` for $k = 1$). Which $k$ does each criterion prefer? Why might they disagree?

In [ ]:
ks = np.arange(1, 8)
bics, sils = [], []

### BEGIN SOLUTION
for k in ks:
    gmm_k = GaussianMixture(n_components=k, n_init=5, random_state=0).fit(X)
    bics.append(gmm_k.bic(X))
    sils.append(silhouette_score(X, gmm_k.predict(X)) if k > 1 else np.nan)
### END SOLUTION

k_bic, k_sil = ks[np.argmin(bics)], ks[np.nanargmax(sils)]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ks, bics, "o-", ms=7)
ax1.axvline(k_bic, color="black", ls="--", lw=1)
ax1.set(xlabel="k", ylabel="BIC", xticks=ks, title=f"BIC: minimum at k = {k_bic}")
ax2.plot(ks, sils, "o-", ms=7)
ax2.axvline(k_sil, color="black", ls="--", lw=1)
ax2.set(xlabel="k", ylabel="mean silhouette", xticks=ks, title=f"Silhouette: maximum at k = {k_sil}")
fig.tight_layout();

Let's look at both answers. To make the components easy to talk about, we sort them by colour and then by density.

In [ ]:
def fit_sorted_gmm(X, k, seed=0):
    # fit a GMM and relabel its components: blue before red, then low before high density
    g = GaussianMixture(n_components=k, n_init=5, random_state=seed).fit(X)
    order = np.lexsort((g.means_[:, 1], g.means_[:, 0] > 0))   # blue before red, then by density
    relabel = np.empty(k, dtype=int)
    relabel[order] = np.arange(k)
    return g, relabel[g.predict(X)], order

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, k in zip(axes, [k_sil, k_bic]):
    g, lab, order = fit_sorted_gmm(X, k)
    for j in range(k):
        m = lab == j
        ax.scatter(d.logSigma[m], d.gr[m], s=4, color=f"C{j}", alpha=0.6, linewidth=0)
    ax.set(xlim=(-1, 3), ylim=(0, 1), xlabel=r"$\log\,\Sigma_5$ [Mpc$^{-2}$]", title=f"GMM, k = {k}")
axes[0].set_ylabel("g − r")
fig.tight_layout()

## 4. Are the populations stable?

BIC and silhouette disagree, so let's use the third tool from the last notebook: **bootstrap stability**. If a clustering reflects real structure in the data, it should come back when we resample the galaxies.

### ✏️ Exercise 3 — Bootstrap stability of the GMM

Adapt the `stability` function from the last notebook to Gaussian mixtures: fit a reference `GaussianMixture(n_components=k, n_init=5, random_state=0)` to all of `X`; then, for each bootstrap, fit a new mixture (`n_init=3, random_state=b`) to a resampled `X[idx]`, predict labels for **all** galaxies, and compute the ARI with the reference. Return the array of ARI values.

In [ ]:
def stability(X, k, n_boot=15, seed=0):
    rng = np.random.default_rng(seed)
    ### BEGIN SOLUTION
    ref = GaussianMixture(n_components=k, n_init=5, random_state=0).fit(X).predict(X)
    scores = []
    for b in range(n_boot):
        idx = rng.choice(len(X), len(X), replace=True)
        new = GaussianMixture(n_components=k, n_init=3, random_state=b).fit(X[idx]).predict(X)
        scores.append(adjusted_rand_score(ref, new))
    return np.array(scores)
    ### END SOLUTION

ks_stab = np.arange(2, 7)
stab = {k: stability(X, k) for k in ks_stab}

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.boxplot([stab[k] for k in ks_stab], positions=ks_stab, widths=0.5, showfliers=False)
jitter = np.random.default_rng(1)
for k in ks_stab:
    ax.scatter(k + jitter.uniform(-0.15, 0.15, len(stab[k])), stab[k], s=10, color="black", alpha=0.5, zorder=3)
    print(f"k = {k}:  median ARI = {np.median(stab[k]):.3f},  min = {stab[k].min():.2f}")
ax.set(xlabel="k", ylabel="ARI vs. full-data fit", xticks=ks_stab, ylim=(0, 1.05),
       title="Bootstrap stability of the Gaussian mixture");

## 5. Validation on the sky

We never told the algorithm where Coma and Abell 1367 are. If the "red and dense" population is physically meaningful, its members should **trace the clusters and groups on the sky** — a property we did not cluster on.

**From here on we fix $K = 4$.** BIC and silhouette (Exercise 2) are nearly tied between $k = 4$ and $k = 5$ — the BIC values differ by only a fraction of a percent, so depending on your platform or `scikit-learn` version you may have gotten either one from the exercise. **We use $K = 4$ explicitly from here on, not whatever `k_bic` happened to evaluate to, so that everyone's populations, sky maps and structures match.**

In [ ]:
K = 4   # NOTE: fixed explicitly (NOT k_bic) -- BIC is nearly tied between k=4 and k=5, see Exercise 2
gmm, labels, order = fit_sorted_gmm(X, K)
d["pop"] = labels
means = scaler.inverse_transform(gmm.means_[order])
RED_DENSE = int(np.argmax((means[:, 0] - means[:, 0].mean())/means[:, 0].std()
                          + (means[:, 1] - means[:, 1].mean())/means[:, 1].std()))

print(f"{'population':>10s} {'g-r':>6s} {'logSigma5':>10s} {'N':>6s}")
for j in range(K):
    tag = "  <- red & dense" if j == RED_DENSE else ""
    print(f"{j:10d} {means[j, 0]:6.2f} {means[j, 1]:10.2f} {np.sum(labels == j):6d}{tag}")

fig, axes = plt.subplots(1, K, figsize=(5*K, 4.8), sharey=True)
for j, ax in enumerate(np.atleast_1d(axes)):
    ax.scatter(d.ra, d.dec, s=2, color="lightgray", linewidth=0)
    m = labels == j
    ax.scatter(d.ra[m], d.dec[m], s=4, color=f"C{j}", linewidth=0)
    sky_axes(ax, f"population {j}: g−r = {means[j, 0]:.2f}, log Σ5 = {means[j, 1]:.1f}")
    if j > 0:
        ax.set_ylabel("")
fig.tight_layout();

### ✏️ Exercise 4 — Quantify it

Select the galaxies within `R_DEG` = 1° of Coma's centre (about 1.8 Mpc). Use the Euclidean distance formula $\sqrt{(\Delta\mathrm{RA})^2 + (\Delta\mathrm{Dec})^2}$ (ignoring the sky's curvature over such a small separation). What fraction of the selected galaxies belong to the red-and-dense population `RED_DENSE`? Compare to the fraction over the whole slice.

In [ ]:
R_DEG = 1.0
is_rd = d["pop"].to_numpy() == RED_DENSE

### BEGIN SOLUTION
ra0, dec0 = CLUSTERS["Coma"]
sep = np.hypot(d.ra - ra0, d.dec - dec0).to_numpy()
near = sep < R_DEG
### END SOLUTION

print(f"Coma       : {near.sum():4d} galaxies within {R_DEG} deg, {is_rd[near].mean():.0%} of them red & dense")
print(f"whole slice: {is_rd.mean():.0%} red & dense")


## 6. From populations to structures

The mixture model tells us **which kind** of galaxy each object is, not **which structure** it belongs to. To split the red-and-dense population into individual clusters and groups, we now cluster the **positions** of those members with DBSCAN. Here position is the quantity of interest, and it has a natural metric, so we do *not* standardise it: we work with angular distances converted to Mpc, just as for $\Sigma_5$.

With `min_samples = 5`, `eps` plays the role of a *linking length*: galaxies closer than `eps` are "friends", and friends of friends end up in the same structure. This is how friends-of-friends group finders work.

### ✏️ Exercise 5 — Find the structures

1. Make the k-distance plot for the red-and-dense members: the sorted distance (in Mpc) to the 5th nearest neighbour. (You already computed exactly this in Exercise 1 — but now only among the members.)
2. Choose `EPS_MPC` from the plot and run `DBSCAN(eps=EPS_MPC / D_C, min_samples=5, metric="haversine")` on the members' coordinates. (Dividing by $D_C$ converts Mpc back to radians.)

The cell then lists the largest structures and plots them. Which two are the largest? Try a larger and a smaller `EPS_MPC`: what merges, and what disappears?

In [ ]:
members = d[is_rd].reset_index(drop=True)
coords_m = np.deg2rad(members[["dec", "ra"]].to_numpy())

### BEGIN SOLUTION
dist_m, _ = NearestNeighbors(n_neighbors=6, metric="haversine").fit(coords_m).kneighbors(coords_m)
kdist = np.sort(dist_m[:, 5] * D_C)
EPS_MPC = 0.4
struct = DBSCAN(eps=EPS_MPC / D_C, min_samples=5, metric="haversine").fit_predict(coords_m)
### END SOLUTION

members["structure"] = struct
sizes = members[members.structure >= 0].groupby("structure").agg(
    N=("ra", "size"), ra=("ra", "mean"), dec=("dec", "mean"), z=("z", "median")
).sort_values("N", ascending=False)
print(f"{len(sizes)} structures with eps = {EPS_MPC} Mpc; {np.mean(struct == -1):.0%} of members are unassigned")
print(sizes.head(8).round(3).to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw=dict(width_ratios=[1, 1.35]))
ax1.plot(kdist)
ax1.axhline(EPS_MPC, color="C1", ls="--", lw=1.5, label=f"eps = {EPS_MPC} Mpc")
ax1.set(xlabel="red & dense members, sorted", ylabel="distance to 5th neighbour [Mpc]",
        title="k-distance plot", yscale="log")
ax1.legend()

ax2.scatter(d.ra, d.dec, s=2, color="lightgray", linewidth=0)
top = sizes.index[:8]
for rank, s_id in enumerate(top):
    m = members.structure == s_id
    ax2.scatter(members.ra[m], members.dec[m], s=6, color=f"C{rank}", linewidth=0)
sky_axes(ax2, f"DBSCAN on red & dense members: the {len(top)} largest structures")
fig.tight_layout();